In [5]:
from dataclasses import dataclass


@dataclass
class Instruction:
    name: str
    destination: str
    sources: list


instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),
    Instruction("I2", "R4", ["R5", "R6"]),
    Instruction("I3", "R7", ["R1", "R4"]),
    Instruction("I4", "R2", ["R5", "R8"]),
    Instruction("I5", "R4", ["R7", "R2"]),
    Instruction("I6", "R7", ["R8", "R9"]),
]


# BAGIAN ANGGOTA 1: Fungsi Dependensi RAW
def has_raw_dependency(inst1, inst2):
    if inst1.destination in inst2.sources:
        return True
    return False


# BAGIAN ANGGOTA 2: Fungsi Dependensi WAR
def has_war_dependency(inst1, inst2):
    if inst2.destination in inst1.sources:
        return True
    return False


def has_dependency(inst1, inst2):
    return has_war_dependency(inst1, inst2)


def show_dependencies(instructions):
    print("\n" + "=" * 50)
    print("DEPENDENCY ANALYSIS (WAR)")
    print("=" * 50)
    found = False
    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            if has_war_dependency(instructions[i], instructions[j]):
                print(
                    f"{instructions[j].name} depends on {instructions[i].name} "
                    f"[WAR - Register: {instructions[j].destination}]"
                )
                found = True
    if not found:
        print("No dependencies found.")


def sequential_execution(instructions):
    print("\n" + "=" * 50)
    print("SEQUENTIAL EXECUTION")
    print("=" * 50)
    for cycle, inst in enumerate(instructions, start=1):
        print(f"Cycle {cycle}: Executing {inst.name} ({inst.destination} = {' + '.join(inst.sources)})")
    print(f"\nTotal Sequential Cycles: {len(instructions)}")
    return len(instructions)


def parallel_execution(instructions):
    print("\n" + "=" * 50)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION (WAR)")
    print("=" * 50)
    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1
        current_cycle = []
        for instruction in remaining:
            dependencies_satisfied = True
            for previous in instructions:
                if previous.name == instruction.name:
                    break
                if has_dependency(previous, instruction) and previous.name not in completed:
                    dependencies_satisfied = False
                    break
            if dependencies_satisfied:
                current_cycle.append(instruction)

        print(f"\nCycle {cycle}:")
        for instruction in current_cycle:
            print(f"  -> {instruction.name}: {instruction.destination} = {' + '.join(instruction.sources)}")
        for instruction in current_cycle:
            completed.add(instruction.name)
            remaining.remove(instruction)

    print(f"\nTotal ILP Cycles: {cycle}")
    return cycle


def main():
    print("\n" + "*" * 60)
    print("      INSTRUCTION-LEVEL PARALLELISM SIMULATOR")
    print("*" * 60)
    print("\nInstruction Set:")
    for inst in instructions:
        print(f"{inst.name}: {inst.destination} = {' + '.join(inst.sources)}")

    show_dependencies(instructions)
    seq_cycles = sequential_execution(instructions)
    ilp_cycles = parallel_execution(instructions)

    speedup = seq_cycles / ilp_cycles
    efficiency = (speedup / len(instructions)) * 100
    print("\n" + "=" * 50)
    print("PERFORMANCE COMPARISON (WAR)")
    print("=" * 50)
    print(f"Sequential Cycles : {seq_cycles}")
    print(f"ILP Cycles        : {ilp_cycles}")
    print(f"Speedup           : {speedup:.2f}x")
    print(f"ILP Efficiency    : {efficiency:.2f}%")


if __name__ == "__main__":
    main()


************************************************************
      INSTRUCTION-LEVEL PARALLELISM SIMULATOR
************************************************************

Instruction Set:
I1: R1 = R2 + R3
I2: R4 = R5 + R6
I3: R7 = R1 + R4
I4: R2 = R5 + R8
I5: R4 = R7 + R2
I6: R7 = R8 + R9

DEPENDENCY ANALYSIS (WAR)
I4 depends on I1 [WAR - Register: R2]
I5 depends on I3 [WAR - Register: R4]
I6 depends on I5 [WAR - Register: R7]

SEQUENTIAL EXECUTION
Cycle 1: Executing I1 (R1 = R2 + R3)
Cycle 2: Executing I2 (R4 = R5 + R6)
Cycle 3: Executing I3 (R7 = R1 + R4)
Cycle 4: Executing I4 (R2 = R5 + R8)
Cycle 5: Executing I5 (R4 = R7 + R2)
Cycle 6: Executing I6 (R7 = R8 + R9)

Total Sequential Cycles: 6

INSTRUCTION-LEVEL PARALLEL EXECUTION (WAR)

Cycle 1:
  -> I1: R1 = R2 + R3
  -> I2: R4 = R5 + R6
  -> I3: R7 = R1 + R4

Cycle 2:
  -> I4: R2 = R5 + R8
  -> I5: R4 = R7 + R2

Cycle 3:
  -> I6: R7 = R8 + R9

Total ILP Cycles: 3

PERFORMANCE COMPARISON (WAR)
Sequential Cycles : 6
ILP Cycles        :